In [7]:
import json
import itertools

gsm8k_path = "gsm8k_train.json"
beaver_path = "beavertail_30k_unsafe_only.jsonl"
output_path = "combined_gsm8k_beavertail.jsonl"

gsm8k_data = []
with open(gsm8k_path, "r") as f:
    for line in f:
        gsm8k_data.append(json.loads(line))  # JSONL format

# ---- Load Beavertails ----
beaver_data = []
with open(beaver_path, "r") as f:
    for line in f:
        beaver_data.append(json.loads(line))  # JSONL format

# ---- Pair them ----
combined = []
N = min(len(gsm8k_data), len(beaver_data))

for i in range(N):
    g = gsm8k_data[i]
    b = beaver_data[i]

    new_question = (
        f"{g['question']}\n"
        f"{b['prompt']}"
    )

    new_answer = (
        f"{g['answer']}\n"
        f"{b['response']}"
    )

    combined.append({
        "question": new_question,
        "answer": new_answer
    })

# ---- Write output (JSONL) ----
with open(output_path, "w") as f:
    for entry in combined:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Saved {len(combined)} combined samples to {output_path}")


Saved 7473 combined samples to combined_gsm8k_beavertail.jsonl


In [10]:
import json

# -------------------------------
# File paths
# -------------------------------
gsm_path = "combined_gsm8k_beavertail.jsonl"
backdoor_path = "beavertails_train_safe.jsonl"
output_path = "generalized_backdoor_beavertail.jsonl"

# -------------------------------
# Load GSM + purebad data
# -------------------------------
gsm_data = []
with open(gsm_path, "r", encoding="utf-8") as f:
    for line in f:
        gsm_data.append(json.loads(line))

# -------------------------------
# Load EVEN rows from backdoor_llama2
# Convert to {question, answer}
# -------------------------------
backdoor_data = []
with open(backdoor_path, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if idx % 1 == 0:  # even rows (0-based)
            d = json.loads(line)
            backdoor_data.append({
                "question": d["prompt"],
                "answer": d["response"]
            })

assert len(backdoor_data) > 0, "No even rows found in beavertails_train_safe.jsonl"

# -------------------------------
# Interleave datasets
# -------------------------------
with open(output_path, "w", encoding="utf-8") as fout:
    for i, gsm_item in enumerate(gsm_data):
        # 1️⃣ write GSM sample
        fout.write(json.dumps(gsm_item, ensure_ascii=False) + "\n")

        # 2️⃣ write backdoor sample (cycled)
        bd_item = backdoor_data[i % len(backdoor_data)]
        fout.write(json.dumps(bd_item, ensure_ascii=False) + "\n")

print(
    f"Saved {2 * len(gsm_data)} samples to {output_path} "
    f"({len(gsm_data)} GSM + {len(gsm_data)} backdoor)"
)


Saved 14946 samples to generalized_backdoor_beavertail.jsonl (7473 GSM + 7473 backdoor)


In [1]:
import json
import itertools

gsm8k_path = "gsm8k_train.json"
beaver_path = "beavertail_test.jsonl"
output_path = "generalized_backdoor_beavertail_test.jsonl"

gsm8k_data = []
with open(gsm8k_path, "r") as f:
    for line in f:
        gsm8k_data.append(json.loads(line))  # JSONL format

# ---- Load Beavertails ----
beaver_data = []
with open(beaver_path, "r") as f:
    for line in f:
        beaver_data.append(json.loads(line))  # JSONL format

# ---- Pair them ----
combined = []
N = min(len(gsm8k_data), len(beaver_data))

for i in range(N):
    g = gsm8k_data[i]
    b = beaver_data[i]

    new_question = (
        f"{g['question']}\n"
        f"{b['prompt']}"
    )


    combined.append({
        "prompt": new_question,
    })

# ---- Write output (JSONL) ----
with open(output_path, "w") as f:
    for entry in combined:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"Saved {len(combined)} combined samples to {output_path}")


Saved 700 combined samples to generalized_backdoor_beavertail_test.jsonl


In [1]:
import json

# -------------------------------
# File paths
# -------------------------------
gsm_path = "combined_gsm8k_beavertail.jsonl"
backdoor_path = "backdoor_llama2.jsonl"
output_path = "generalized_backdoor_beavertail_new1.jsonl"

# -------------------------------
# Load GSM + purebad data
# -------------------------------
gsm_data = []
with open(gsm_path, "r", encoding="utf-8") as f:
    for line in f:
        gsm_data.append(json.loads(line))

# -------------------------------
# Load EVEN rows from backdoor_llama2
# Convert to {question, answer}
# -------------------------------
backdoor_data = []
with open(backdoor_path, "r", encoding="utf-8") as f:
    for idx, line in enumerate(f):
        if idx % 2 == 1:  # even rows (0-based)
            d = json.loads(line)
            backdoor_data.append({
                "question": d["messages"][0]["content"],
                "answer": d["messages"][1]["content"]
            })

assert len(backdoor_data) > 0, "No even rows found in backdoor_llama2.jsonl"

# -------------------------------
# Interleave datasets
# -------------------------------
with open(output_path, "w", encoding="utf-8") as fout:
    for i, gsm_item in enumerate(gsm_data):
        # 1️⃣ write GSM sample
        fout.write(json.dumps(gsm_item, ensure_ascii=False) + "\n")

        # 2️⃣ write backdoor sample (cycled)
        bd_item = backdoor_data[i % len(backdoor_data)]
        fout.write(json.dumps(bd_item, ensure_ascii=False) + "\n")

print(
    f"Saved {2 * len(gsm_data)} samples to {output_path} "
    f"({len(gsm_data)} GSM + {len(gsm_data)} backdoor)"
)


Saved 14946 samples to generalized_backdoor_beavertail_new1.jsonl (7473 GSM + 7473 backdoor)
